In [ ]:
!nvcc --version

/bin/bash: line 1: nvcc: command not found


In [ ]:
!apt-get update
!apt-get install -y nvidia-cuda-toolkit

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,015 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:14 http:/

In [ ]:
%%writefile copy_sm_warp.cu
#include <stdio.h>

__global__ void copyKernel(float *A, float *B, int N) {

    int global_id = blockIdx.x * blockDim.x + threadIdx.x;

    // Warp ID inside block
    int warp_id = threadIdx.x / 32;

    // Thread ID inside warp
    int lane_id = threadIdx.x % 32;

    // Get SM ID (inline PTX)
    int smid;
    asm("mov.u32 %0, %smid;" : "=r"(smid));

    if (global_id < N) {
        A[global_id] = B[global_id];
    }

    // Print only first thread of each warp to reduce output
    if (lane_id == 0 && global_id < N) {
        printf("Block %d | SM %d | Warp %d | Global Thread %d\n",
               blockIdx.x, smid, warp_id, global_id);
    }
}

int main() {

    const int N = 8192;   // total threads (your requirement)
    const int THREADS = 256;   // block size → 8 warps
    const int BLOCKS = N / THREADS;  // 8192 / 256 = 32 blocks

    float *h_A, *h_B;
    float *d_A, *d_B;

    size_t size = N * sizeof(float);

    // Allocate host memory
    h_A = (float*)malloc(size);
    h_B = (float*)malloc(size);

    // Initialize input
    for (int i = 0; i < N; i++) {
        h_B[i] = i * 1.0f;
    }

    // Allocate device memory
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);

    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Launch kernel
    copyKernel<<<BLOCKS, THREADS>>>(d_A, d_B, N);

    cudaMemcpy(h_A, d_A, size, cudaMemcpyDeviceToHost);

    cudaDeviceSynchronize();

    // Print a few results
    printf("\nSample Output Check:\n");
    for (int i = 0; i < 5; i++) {
        printf("A[%d] = %f\n", i, h_A[i]);
    }

    cudaFree(d_A);
    cudaFree(d_B);
    free(h_A);
    free(h_B);

    return 0;
}

Writing copy_sm_warp.cu


In [ ]:
!nvcc copy_sm_warp.cu -o copy_sm_warp

/bin/bash: line 1: nvcc: command not found
